In [2]:
!pip install pandas

  Using cached pytz-2025.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.1-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 8.2 MB/s eta 0:00:00:00:0100:01
Using cached pytz-2025.1-py2.py3-none-any.whl (507 kB)
Using cached tzdata-2025.1-py2.py3-none-any.whl (346 kB)


In [13]:
!pip install datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached datasets-3.3.2-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached fsspec-2024.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached aiohappyeyeballs-2.4.6-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)
  Using cached attrs-25.1.0-py3-none-any.whl.metadata (10 kB)
Using cached datasets-3.3.2-py3-none-any.whl (485 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached fsspec-2024.12.0-py3-none-any.whl (183 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.3 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 1.0 MB/s eta 0:00:0000:0100:02
Using cached aiohappyeyeballs-2.4.6-py3-none-any.whl (14 kB)
Using cached aiosignal-1.3.2-py2.py3-none-any.whl (7.6 kB)
Using cached attrs-25.1.0-py3-none-any.whl (63 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.2.0
    Uninstalling fsspec

In [1]:
import pandas as pd

df = pd.read_table('./data/2019/caption.txt', header=None)  #2014
df.rename(columns={0: "file_name", 1: "text"}, inplace=True)
df['file_name']= df['file_name'].apply(lambda x: x+'.jpg')
df = df.dropna()
df.head()


,file_name,text
0,UN19_1023_em_326.jpg,n \times 3
1,UN19_1039_em_567.jpg,x = \cos L t
2,UN19_1038_em_545.jpg,- \frac { 3 + z ^ { 2 } } { 8 } - \frac { ( 3 ...
3,UN19_1022_em_304.jpg,\frac { 1 } { 8 } ( n + 2 ) ( n + 4 )
4,UN19_1003_em_30.jpg,n = - 1 + \sqrt { 1 5 }


In [2]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=490):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text 
        file_name = self.df['file_name'][idx]
        text = self.df['text'][idx]
        # prepare image (i.e. resize + normalize)
        image = Image.open(self.root_dir + file_name).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        # add labels (input_ids) by encoding the text
        labels = self.processor.tokenizer(text, 
                                          padding="max_length", 
                                          max_length=self.max_target_length).input_ids
        # important: make sure that PAD tokens are ignored by the loss function
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
        return encoding

In [3]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
test_dataset = IAMDataset(root_dir='./data/2019/',
                           df=df,
                           processor=processor)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
from torch.utils.data import DataLoader

test_dataloader = DataLoader(test_dataset, batch_size=1)

In [5]:
batch = next(iter(test_dataloader))

In [6]:
for k,v in batch.items():
  print(k, v.shape)

pixel_values torch.Size([1, 3, 384, 384])
labels torch.Size([1, 490])


In [7]:
labels = batch["labels"]
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.batch_decode(labels, skip_special_tokens=True)
label_str

['n \\times 3']

In [8]:
from transformers import VisionEncoderDecoderModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionEncoderDecoderModel.from_pretrained('./checkpoint_eval_2014_small_stage1_new_image/checkpoint-11000')
model.to(device)

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.49.0"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decode

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linea

In [19]:
!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 10.8 MB/s eta 0:00:00a 0:00:01
Using cached joblib-1.4.2-py3-none-any.whl (301 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 1.9 MB/s eta 0:00:0000:0100:01
Using cached threadpoolctl-3.5.0-py3-none-any.whl (18 kB)


In [9]:
import evaluate

cer_metric = evaluate.load("cer")

accuracy_metric = evaluate.load("accuracy")


Using the latest cached version of the module from /home/saphalr/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--cer/9cb90b752d5f15fb41161efdbefd13570adb3f32fa157290d8a55093c47428e1 (last modified on Sat Mar  1 14:28:53 2025) since it couldn't be found locally at evaluate-metric--cer, or remotely on the Hugging Face Hub.


In [14]:
from datasets import load_metric

#cer_metric = load_metric("accuracy")
cer_metric = load_metric("cer")

ImportError: cannot import name 'load_metric' from 'datasets' (/home/saphalr/miniconda3/envs/torch/lib/python3.9/site-packages/datasets/__init__.py)

In [23]:
!pip install ipywidgets --upgrade

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 5.5 MB/s eta 0:00:00a 0:00:01


In [10]:
from tqdm import tqdm
import numpy as np

print("Running evaluation...")

total = 0
pred_label = 0

for batch in tqdm(test_dataloader):
    # predict using generate
    pixel_values = batch["pixel_values"].to(device)
    outputs = model.generate(pixel_values)
    # decode
    pred_str = processor.batch_decode(outputs, skip_special_tokens=True)
    labels = batch["labels"]
    labels[labels == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels, skip_special_tokens=True)

    if pred_str == label_str:
        pred_label += 1
    total += 1

    #pred_str = np.argmax(pred_str)

    # add batch to metric
    cer_metric.add_batch(predictions=pred_str, references=label_str)

Accuracy_score = pred_label/total
final_score = cer_metric.compute()

Running evaluation...


100%|██████████| 1199/1199 [27:57<00:00,  1.40s/it]


In [27]:
#from tqdm.notebook import tqdm
#import numpy as np

#print("Running evaluation...")

#for batch in tqdm(test_dataloader):
    # predict using generate
#    pixel_values = batch["pixel_values"].to(device)
#    outputs = model.generate(pixel_values)

    # decode
#    pred_str = processor.batch_decode(outputs, skip_special_tokens=True)
#    labels = batch["labels"]
#    labels[labels == -100] = processor.tokenizer.pad_token_id
#    label_str = processor.batch_decode(labels, skip_special_tokens=True)

    # add batch to metric
#    cer_metric.add_batch(predictions=pred_str, references=label_str)
#
#final_score = cer_metric.compute()

Running evaluation...


  0%|          | 0/986 [00:00<?, ?it/s]

In [37]:
print("Character error rate on test set(2014):", final_score)

Character error rate on test set(2014): 0.32765843179377013


In [38]:
print("Accuracy rate on test set (2014):", Accuracy_score)

Accuracy rate on test set (2014): 0.2150101419878296


In [40]:
print("Character error rate on test set(2016):", final_score)
print("Accuracy rate on test set (2016):", Accuracy_score)

Character error rate on test set(2016): 0.3206294829247404
Accuracy rate on test set (2016): 0.22057541412380122


In [11]:
print("Character error rate on test set(2019):", final_score)
print("Accuracy rate on test set (2019):", Accuracy_score)

Character error rate on test set(2019): 0.291548203670199
Accuracy rate on test set (2019): 0.23019182652210174


In [12]:
!pip install torchviz

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.3
    Uninstalling sympy-1.13.3:
      Successfully uninstalled sympy-1.13.3


In [24]:
!pip install netron


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 9.2 MB/s eta 0:00:00


In [26]:
import torch
from transformers import VisionEncoderDecoderModel, AutoProcessor
from PIL import Image
import os
import numpy as np

# =============== METHOD 1: ONNX Export for Netron ===============
def visualize_with_netron():
    # Load the TrOCR model
    model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
    processor = AutoProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    
    # Set decoder start token id
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    
    # Prepare dummy input for tracing
    # Use the expected shape of the TrOCR model: [batch_size, channels, height, width]
    dummy_input = torch.zeros(1, 3, 384, 384)
    
    # Create directories if they don't exist
    os.makedirs("model_viz", exist_ok=True)
    
    # Export to ONNX for visualization in Netron
    torch.onnx.export(
        model,
        dummy_input,
        "model_viz/trocr_model.onnx",
        opset_version=12,
        input_names=["pixel_values"],
        output_names=["logits"],
        dynamic_axes={
            "pixel_values": {0: "batch_size"},
            "logits": {0: "batch_size"}
        }
    )
    
    print("Model exported to ONNX format at 'model_viz/trocr_model.onnx'")
    print("You can now open this file with Netron")
    
    # Optional: Open with Netron directly if installed
    try:
        import netron
        netron.start("model_viz/trocr_model.onnx")
        print("Netron viewer started")
    except ImportError:
        print("To view the model, install Netron with: pip install netron")
        print("Or download Netron from: https://github.com/lutzroeder/netron/releases")

In [27]:
visualize_with_netron()

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.49.0"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decode

ValueError: You have to specify either decoder_input_ids or decoder_inputs_embeds